
# CVXPY Tutorial: Convex Optimization in Python

This notebook is a **hands-on tutorial** to learn **CVXPY**, a Python library for modeling and solving **convex optimization** problems.

By the end, you will be able to:
- create **decision variables**
- build **objective functions** (minimize / maximize)
- add **constraints**
- solve the optimization problem
- interpret **solutions** and **dual variables** (shadow prices)
- recognize common convex problem classes: **LP, QP, SOCP**

> **Important:** CVXPY is a *modeling* tool: you write the math in Python, and CVXPY calls a solver (OSQP, ECOS, SCS, …).

---

## Quick prerequisites
- Vectors/matrices, basic calculus
- Convex sets: `x >= 0`, `Ax <= b`, norm balls
- Convex functions: linear, quadratic with PSD matrix, norms

---


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

import cvxpy as cp

np.random.seed(0)



## 1) The CVXPY modeling pattern (always the same)

Every CVXPY model has 5 steps:

1. **Create variables**: `x = cp.Variable(...)`
2. **Write objective**: `cp.Minimize(f(x))` or `cp.Maximize(f(x))`
3. **Write constraints**: a Python list like `[Ax <= b, x >= 0]`
4. **Create the problem**: `prob = cp.Problem(objective, constraints)`
5. **Solve**: `prob.solve()`

Then read results:
- `x.value` (optimal decision)
- `prob.value` (optimal objective value)
- `constraint.dual_value` (dual variables / shadow prices)

Let's start with a tiny example.


In [ ]:

# Example 1: minimize (x-3)^2 subject to x >= 0

x = cp.Variable()  # scalar variable

objective = cp.Minimize(cp.square(x - 3))
constraints = [x >= 0]

prob = cp.Problem(objective, constraints)
prob.solve()

print("status:", prob.status)
print("x* =", x.value)
print("objective value =", prob.value)



### Why is this convex?
- The function $(x-3)^2$ is convex (a parabola).
- The constraint $x \ge 0$ is a convex set (a half-line).
Therefore, the whole problem is convex and has a global optimum.

---

## 2) Linear Programming (LP) example: Diet / cost minimization

**Story:** A school cafeteria wants a low-cost meal plan while meeting nutrition requirements.

Decision variables:
- $x_1, x_2, x_3 \ge 0$: quantities of 3 foods.

Objective:
- minimize total cost.

Constraints:
- enough protein and calories.

This is a **Linear Program** because both objective and constraints are linear.


In [ ]:

# Food costs (USD per serving)
c = np.array([0.8, 1.1, 0.6])

# Nutrient matrix: rows = nutrients, cols = foods
# protein (g), calories (kcal)
A = np.array([
    [10,  20,  5],   # protein
    [200, 250, 150]  # calories
], dtype=float)

b = np.array([60, 1800], dtype=float)  # minimum requirements

x = cp.Variable(3, nonneg=True)

objective = cp.Minimize(c @ x)  # linear objective
constraints = [A @ x >= b]      # linear inequality

prob = cp.Problem(objective, constraints)
prob.solve()

print("status:", prob.status)
print("x* =", x.value)
print("min cost =", prob.value)

# Dual variables (shadow prices) for nutrient constraints
print("dual for protein constraint:", constraints[0].dual_value[0])
print("dual for calories constraint:", constraints[0].dual_value[1])



### Interpreting dual variables (shadow prices)
The dual variables tell you the **marginal value** of relaxing constraints:
- If the **protein dual** is high, protein is “expensive” in the optimization sense.
- If a constraint is not tight, its dual is usually close to 0 (complementary slackness).

---

## 3) Quadratic Programming (QP): Ridge regression (machine learning)

We fit a linear model $y \approx Xw$ with **L2 regularization**.

Optimization problem:
$
\min_w \frac{1}{n}\|Xw - y\|_2^2 + \lambda \|w\|_2^2
$

This is convex because:
- squared norm is convex
- L2 regularization is convex
- sum of convex functions is convex


In [ ]:

# Create synthetic regression data
n, d = 120, 5
X = np.random.randn(n, d)
w_true = np.array([1.5, -2.0, 0.0, 0.7, 1.0])
y = X @ w_true + 0.5*np.random.randn(n)

lam = 0.5

w = cp.Variable(d)

objective = cp.Minimize((1/n)*cp.sum_squares(X @ w - y) + lam*cp.sum_squares(w))
prob = cp.Problem(objective)
prob.solve(solver=cp.OSQP)

print("status:", prob.status)
print("w* =", w.value.round(3))
print("objective value =", prob.value)



### Plot: True weights vs estimated weights


In [ ]:

plt.figure()
plt.stem(w_true, label="true", use_line_collection=True)
plt.stem(w.value, label="estimated", use_line_collection=True)
plt.legend()
plt.xlabel("weight index")
plt.title("Ridge regression via convex optimization (CVXPY)")
plt.show()



## 4) Portfolio optimization (finance): Mean–variance with constraints

We choose asset weights $w$ to minimize risk for a target return.

Problem (one standard version):
$$
\min_w\; w^\top \Sigma w
\quad \text{s.t.}\quad \mu^\top w \ge r_{\text{target}},\; \sum_i w_i = 1,\; w \ge 0
$$

This is a convex QP when $\Sigma$ is positive semidefinite.


In [ ]:

n_assets = 6
mu = np.random.uniform(0.05, 0.20, size=n_assets)

# Build a PSD covariance matrix
A = np.random.randn(n_assets, n_assets)
Sigma = (A.T @ A) / 20.0

r_target = 0.12

w = cp.Variable(n_assets)

objective = cp.Minimize(cp.quad_form(w, Sigma))
constraints = [
    mu @ w >= r_target,
    cp.sum(w) == 1,
    w >= 0
]

prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.OSQP)

print("status:", prob.status)
print("weights:", w.value.round(3))
print("risk (variance):", prob.value)
print("expected return:", mu @ w.value)



### Exercise 
1. Change `r_target` and see how risk changes.
2. Add a constraint: "no more than 40% in one asset": $w_i \le 0.4$.
3. Plot the efficient frontier by sweeping `r_target`.

A solution template is below.


In [ ]:

# Solution template for plotting a small efficient frontier
targets = np.linspace(mu.min()+0.01, mu.max()-0.01, 20)
risks = []
rets = []

for rt in targets:
    w = cp.Variable(n_assets)
    objective = cp.Minimize(cp.quad_form(w, Sigma))
    constraints = [mu @ w >= rt, cp.sum(w) == 1, w >= 0, w <= 0.4]
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.OSQP, warm_start=True, verbose=False)
    if prob.status == "optimal":
        risks.append(prob.value)
        rets.append(mu @ w.value)

plt.figure()
plt.plot(risks, rets, marker="o")
plt.xlabel("Risk (variance)")
plt.ylabel("Expected return")
plt.title("Efficient frontier (with long-only + max weight constraint)")
plt.show()



## 5) L1 regularization (LASSO) and sparsity

LASSO:
$
\min_w \frac{1}{n}\|Xw-y\|_2^2 + \lambda \|w\|_1
$

This is convex (quadratic + norm1). It often produces **sparse** solutions (many zeros).


In [ ]:

lam = 0.3
w = cp.Variable(d)
objective = cp.Minimize((1/n)*cp.sum_squares(X @ w - y) + lam*cp.norm1(w))
prob = cp.Problem(objective)
prob.solve(solver=cp.OSQP)

print("w* =", w.value.round(3))
print("number of ~zeros:", np.sum(np.abs(w.value) < 1e-3))



## 6) SOCP example: Minimizing maximum absolute error (Chebyshev regression)

We want a model that minimizes the **worst-case** prediction error:
$$
\min_{w,t}\; t\quad\text{s.t.}\quad |Xw-y| \le t\mathbf{1}
$$

This is convex and is a classic **minimax** formulation.


In [ ]:

w = cp.Variable(d)
t = cp.Variable(nonneg=True)

constraints = [
    X @ w - y <= t,
    -(X @ w - y) <= t
]

objective = cp.Minimize(t)
prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.ECOS)

print("t* (max abs error) =", t.value)
print("w* =", w.value.round(3))



## 7) Common CVXPY mistakes (and how to avoid them)

### DCP rules (Disciplined Convex Programming)
CVXPY checks whether your problem is convex using DCP rules.
Typical errors happen when you:
- multiply two variables (non-convex): `x*y`
- maximize a convex function
- use `cp.quad_form(w, Q)` with a non-PSD matrix `Q`
- write `cp.sqrt(variable)` incorrectly without constraints

Let's see one example and fix it.


In [ ]:

# Non-convex example (DO NOT DO THIS): minimize x*y is not convex
x = cp.Variable()
y = cp.Variable(nonneg=True)

try:
    prob = cp.Problem(cp.Minimize(x*y), [x >= 1, y >= 1])
    prob.solve()
except Exception as e:
    print("Expected DCP error:", e)



### Fixing the non-convex idea
If you want something like minimizing a product, you often:
- take logs (geometric programming) **or**
- use a convex surrogate **or**
- reformulate with additional variables (when possible)

For this course, remember:
> **Convex optimization is powerful because we can guarantee global optimality.**  
> If a model is non-convex, we need different tools (local methods, heuristics, etc.).

---

## 8) Summary cheat-sheet

- **LP**: linear objective + linear constraints  
- **QP**: quadratic (convex) objective + linear constraints  
- **SOCP**: norms in constraints/objective (e.g., \(\|Ax+b\|_2 \le t\))  
- **Dual values**: interpret marginal values of constraints  
- CVXPY requires **DCP** (convexity check)

---
